# Ply GPU worker on Kaggle (T4)

Processes captures sent from the Ply web app (ricco.codes/ply/capture.html). It polls the Ply API, claims the oldest queued
job, runs the pipeline, uploads `scene.spz` + floor data, and repeats. It stops after 45 minutes with no work, so GPU hours
are not burned idling.

**Inputs (Datasets, both private):**
- `ply-pipeline`: the repo's `pipeline/` folder.
- `ply-worker-config`: `config.json` with `{"api": "https://ply-api.<you>.workers.dev", "token": "<WORKER_TOKEN>"}`.
- optional `ply-wheels`: prebuilt CUDA wheels from an earlier run (saves ~15 min).

**Settings:** Accelerator GPU T4 x1 (not P100), Internet on. Run all, or push with `kaggle kernels push` (runs unattended).

In [ ]:
import glob, os, subprocess, sys, json, shutil, pathlib
import time, urllib.request
NOTEBOOK_T0 = time.time()
def early_hello(note):  # before anything heavy, so a stuck machine is visible on the status page and to the supervisor
    try:
        cfgp = next(iter(glob.glob("/kaggle/input/**/config.json", recursive=True)), None)
        c = json.load(open(cfgp))
        urllib.request.urlopen(urllib.request.Request(c["api"].rstrip("/") + "/v1/worker/hello", data=json.dumps({"worker": "kaggle-boot", "note": note}).encode(),
            method="POST", headers={"authorization": "Bearer " + c["token"], "content-type": "application/json", "user-agent": "ply-worker/1.0 (+https://ricco.codes/ply)"}), timeout=15)
    except Exception as e:
        print("early hello failed:", e)
early_hello("installing: notebook started, checking GPU")
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True, timeout=90).stdout.strip()
except FileNotFoundError:  # no NVIDIA driver at all: a CPU machine (rehearsal mode below)
    gpu = ""
except subprocess.TimeoutExpired:
    early_hello("installing: GPU not responding (nvidia-smi timed out)")
    raise SystemExit("nvidia-smi hung for 90 s: this machine's GPU is not responding. Re-run to get another machine.")
print("GPU:", gpu or "none")
# no GPU = CPU rehearsal mode: install the CPU builds, skip the CUDA-only compiles, and only claim rehearsal jobs
# (capture ids starting with "rehearsal-"), whose args carry --train-dry-run. Costs no GPU quota.
REHEARSAL = not gpu
if REHEARSAL:
    print("REHEARSAL MODE (no GPU): only rehearsal jobs will be claimed")
if "P100" in gpu:
    raise SystemExit("P100 (sm_60) is not supported by torch 2.11 cu128 wheels. Switch the accelerator to T4.")
ARCH = "7.5" if ("T4" in gpu or REHEARSAL) else ("8.9" if ("L4" in gpu or "4090" in gpu) else "7.5;8.0;8.6;8.9")
print("TORCH_CUDA_ARCH_LIST =", ARCH, "| python", sys.version.split()[0])

In [ ]:
# pipeline source -> /kaggle/working/pipeline (writable, editable install)
def find_input(rel, must=None):
    """Kaggle has moved dataset mounts before (/kaggle/input/<slug>/ vs nested owner folders); search instead of assuming."""
    hits = [p for p in glob.glob(f"/kaggle/input/**/{rel}", recursive=True) if must is None or must(p)]
    if not hits:
        for root, dirs, files in os.walk("/kaggle/input"):
            depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
            if depth <= 3:
                print("  ", root, sorted(files)[:8])
    return sorted(hits, key=len)[0] if hits else None
src = find_input("pyproject.toml", lambda p: os.path.isdir(os.path.join(os.path.dirname(p), "plypipe")))
assert src, "Add the Ply repo's pipeline/ folder as a Dataset named ply-pipeline"
PIPE = "/kaggle/working/pipeline"
if not os.path.exists(PIPE):
    shutil.copytree(os.path.dirname(src), PIPE, ignore=shutil.ignore_patterns(".venv", "work", "__pycache__"))
wheels = find_input("gsplat-*.whl")
env = dict(os.environ, TORCH_CUDA_ARCH_LIST=ARCH, PLY_GSPLAT_DIR="/kaggle/working/gsplat", MAX_JOBS="4", PYTHON=sys.executable, **({"PLY_CPU": "1"} if REHEARSAL else {}))
if wheels:
    env["PLY_WHEELS"] = os.path.dirname(wheels); print("using prebuilt wheels from", env["PLY_WHEELS"])
else:
    env["PLY_SAVE_WHEELS"] = "/kaggle/working/wheels"; print("building CUDA wheels; they will be saved to /kaggle/working/wheels")
print("nvcc:", subprocess.run(["bash", "-c", "nvcc --version | tail -2"], capture_output=True, text=True).stdout.strip())
# tell the Ply API what we're doing while the CUDA build runs (the status page shows it; Kaggle hides live logs)
import time, urllib.request
cfg_path = find_input("config.json", lambda p: "token" in open(p).read())
CFG = json.load(open(cfg_path)) if cfg_path else None
WORKER_NAME = "kaggle-cpu-rehearsal" if REHEARSAL else "kaggle-" + gpu.split(",")[0].replace(" ", "-")
def hello(note):
    if not CFG:
        return
    try:
        body = json.dumps({"worker": WORKER_NAME, "note": note[:190]}).encode()
        urllib.request.urlopen(urllib.request.Request(CFG["api"].rstrip("/") + "/v1/worker/hello", data=body, method="POST", headers={
            "authorization": "Bearer " + CFG["token"], "content-type": "application/json", "user-agent": "ply-worker/1.0 (+https://ricco.codes/ply)"}), timeout=15)
    except Exception as e:
        print("hello failed:", e)
# install_fast.sh: prebuilt gsplat 1.5.3 in a Python 3.10 venv for training (minutes). install_gpu.sh compiles
# gsplat main + PPISP (over an hour on Kaggle) unless prebuilt wheels are attached as the ply-wheels dataset.
INSTALL_SCRIPT = "install_gpu.sh" if wheels else "install_fast.sh"
env.update(PLY_GSPLAT_DIR="/kaggle/working/gsplat153" if not wheels else env["PLY_GSPLAT_DIR"], PLY_VENV="/kaggle/working/venv310")
print("install script:", INSTALL_SCRIPT)
t0 = time.time()
INSTALL_LIMIT = 50 * 60   # a CUDA build that runs longer than this is stuck: fail fast instead of burning GPU quota (6 h/week)
hello("installing: starting")
with open("/kaggle/working/install.log", "w") as f:  # full log kept as an output file; the tail is printed if it fails
    proc = subprocess.Popen(["bash", f"{PIPE}/scripts/{INSTALL_SCRIPT}"], env=env, stdout=f, stderr=subprocess.STDOUT)
    while proc.poll() is None:
        time.sleep(20)
        if time.time() - t0 > INSTALL_LIMIT:
            proc.kill(); hello("install timed out after 50 min"); print("install exceeded 50 min, killed"); break
        last = ""
        for line in open("/kaggle/working/install.log", errors="replace").read().splitlines()[::-1]:
            if '"stage":"install"' in line:
                try: last = json.loads(line)["msg"]
                except ValueError: pass
                break
        mins = int((time.time() - t0) // 60)
        print(f"[{mins:>2} min] {last}", flush=True)
        hello(f"installing ({mins} min): {last}")
    r = proc
log_tail = open("/kaggle/working/install.log").read()[-6000:]
print(log_tail if r.returncode else log_tail[-800:])
assert r.returncode == 0, "install failed: see the log above (/kaggle/working/install.log)"

In [ ]:
# Node.js for splat-transform (SPZ export)
if shutil.which("npx") is None:
    subprocess.run("curl -fsSL https://nodejs.org/dist/v22.14.0/node-v22.14.0-linux-x64.tar.xz | tar -xJ -C /kaggle/working", shell=True, check=True)
    os.environ["PATH"] = "/kaggle/working/node-v22.14.0-linux-x64/bin:" + os.environ["PATH"]
print(subprocess.run(["node", "-v"], capture_output=True, text=True).stdout)

In [ ]:
import time
# ---- run the worker ----
IDLE_EXIT_MIN = 3 if REHEARSAL else 30
MAX_SESSION_MIN = 180   # hard cap for the whole session; Kaggle GPU quota is 6 h/week
cfg_path = find_input("config.json", lambda p: "token" in open(p).read())
assert cfg_path, "Add the private dataset ply-worker-config with config.json {api, token}"
cfg = json.load(open(cfg_path))
wenv = dict(os.environ, PLY_API=cfg["api"], PLY_WORKER_TOKEN=cfg["token"], PLY_WORKER_NAME=WORKER_NAME, PLY_CLAIM_PREFIX="rehearsal-" if REHEARSAL else "",
            PLY_WORK_DIR="/kaggle/working/work", PLY_GSPLAT_DIR=env["PLY_GSPLAT_DIR"],
            **({"PLY_TRAIN_PYTHON": "/kaggle/working/venv310/bin/python"} if INSTALL_SCRIPT == "install_fast.sh" else {}), PLY_IDLE_EXIT_MIN=str(IDLE_EXIT_MIN),
            PYTHONPATH=PIPE + os.pathsep + os.environ.get("PYTHONPATH", ""))
p = subprocess.Popen([sys.executable, "-m", "plypipe.worker"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=PIPE, env=wenv)
deadline = time.time() + MAX_SESSION_MIN * 60
for line in p.stdout:
    if time.time() > deadline:
        print("session cap reached, stopping worker"); p.kill(); break
    try:
        e = json.loads(line)
        if e.get("event") in ("start", "done", "skip", "warn", "error"):
            print(f"[{e['stage']}] {e['event']} {e.get('msg', '')}"[:300], flush=True)
    except ValueError:
        print(line.rstrip()[:300], flush=True)
print("worker exited with", p.wait())